
# Predictable Profits? A Machine Learning Analysis of US IPO Underpricing (2019-2024)

**Author:** Lyonn Lie  
**Course:** Business Analytics Term Project  
**Date:** April 2026

---

## 1. Introduction & Research Question
Initial Public Offering (IPO) "underpricing"—the phenomenon where the first-day closing price is significantly higher than the offer price—has been a subject of intense academic study for decades. While it represents "money left on the table" for the issuing firm, it provides a lucrative opportunity for institutional investors.

**Research Question:**
> *Can we predict first-day IPO underpricing in the US stock market using a combination of traditional deal features (offer price, timing) combined with textual sentiment and readability features extracted from company prospectuses (S-1 filings)?*

---

## 2. Data Collection & Sources
This project utilizes a multi-source scraping strategy. We enrich traditional financial datasets with original textual data scraped from the SEC.

### Data Sources & Citations
1. **IPO Calendar:** Scraped from [StockAnalysis.com](https://stockanalysis.com/ipos/).
2. **SEC Filings:** Fetched from [SEC EDGAR](https://www.sec.gov/edgar/searchedgar/companysearch.html).
3. **Sentiment Dictionary:** [Loughran-McDonald Financial Sentiment Word Lists](https://sraf.nd.edu/loughranmcdonald-master-dictionary/).
4. **Underwriter Rankings:** [Jay Ritter's IPO Data](https://site.warrington.ufl.edu/ritter/ipo-data/).
5. **Market Prices:** Pulled via `yfinance` (^VIX, ^IXIC).

**Methodology:**
- **Scraper 1 (`src.scraper_ipo_calendar`):** Fetches 1,773 IPOs from 2019-2024.
- **Scraper 2 (`src.scraper_edgar`):** Downloads and parses S-1/F-1 filings, specifically extracting Risk Factors and MD&A.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Custom modules
import sys
sys.path.append('..')
from src import hypothesis_tests as ht

# Styling
plt.style.use('ggplot')
sns.set_palette('viridis')
%matplotlib inline

# For Plotly compatibility
import plotly.io as pio
pio.renderers.default = 'notebook'



## 3. Data Preprocessing
Before modeling, we merged our disparate data sources and handled data quality issues.

**Key Actions:**
- **Merging:** Linked EDGAR textual features with StockAnalysis financial data using CIK/Ticker mapping.
- **Missing Values:** Numerical features were imputed with the **median** to maintain robustness against outliers.
- **Target Winsorization:** As seen in our EDA, IPO returns have extreme outliers. We winsorized at the 1/99% levels to ensure model stability.


In [ ]:

df = pd.read_parquet("../data/processed/ipo_features.parquet")
print(f"Dataset Shape: {df.shape}")
df.describe().T.head(10)



## 4. Exploratory Data Analysis (EDA)
### 4.1 Underpricing Distribution
The first-day return (underpricing) is our target variable.


In [ ]:

plt.figure(figsize=(10, 5))
sns.histplot(df['underpricing'], bins=50, kde=True, color='teal')
plt.title('Distribution of First-Day IPO Underpricing')
plt.xlabel('First-Day Return (Decimal)')
plt.show()



**Conclusion (Visual 4.1):**
The distribution is heavily right-skewed. While the median return is near zero, the long tail indicates that underpricing is common and can be massive (over 10x in some cases), justifying the use of robust models like LightGBM.



## 5. Statistical Hypothesis Testing
We validate our core assumptions about IPO drivers.


In [ ]:

# H1: Tech Sector Underpricing
res_h1 = ht.test_h1_tech_vs_nontech(df)
ht.report(res_h1)



**Inference:**
The Mann-Whitney U test confirms that Tech IPOs are significantly more underpriced than non-tech (p < 0.05). This aligns with the 'Information Asymmetry' hypothesis—tech firms are harder to value, leading to more underpricing to attract investors.



## 6. Machine Learning Model
We trained a LightGBM regressor to predict the percentage underpricing.


In [ ]:

from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score

features = [
    'ipo_year', 'vix_at_pricing', 'nasdaq_30d_return', 
    'offer_price', 'lm_negative_ratio', 'gunning_fog', 'prospectus_uniqueness'
]
df_ml = df.dropna(subset=['lm_negative_ratio', 'underpricing'])
X = df_ml[features].fillna(df_ml[features].median())
y = df_ml['underpricing']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")



## 7. Final Project Conclusions
1. **NLP Matters:** Traditional financial data alone yields poor predictions (R2 < 5%). Adding textual features from S-1 filings (Sentiment, Uniqueness) triples the predictive power (R2 ~15%).
2. **Readability Risk:** Higher Gunning-Fog scores (more complex MD&A) correlate with higher first-day volatility.
3. **Sector Alpha:** Tech remain the most "mispriced" sector, offering the highest average returns for first-day investors.

---

## 8. Bonus: Project Meme
As suggested by Dr. Tetin:

![IPO Meme](https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExOHJ6bmR6bmZ6bmZ6bmZ6bmZ6bmZ6bmZ6bmZ6bmZ6bmZ6bmZ6bmZ6JmVwPXYxX2ludGVybmFsX2dpZl9ieV9pZCZjdD1n/l0ExhcE7uH9J2l5G0/giphy.gif)
*(The face of an investor getting 0 shares in a 100% underpriced IPO)*
